# Energy Map: підготовка даних про застосування ГПВ

Версія навчального ноутбука для репозиторію **Ukraine Electricity Market**.
Основне джерело проєкту — НКРЕКП; відключення використовуються як контекст.
[Джерело — Energy Map](https://energy-map.info/uk/datasets/0f8f9882-1fb2-47c6-81dc-31fbad914f16).

Ноутбук читає **збережений API-знімок**, не витрачає запити та не потребує токена.
Щомісячне завантаження виконує GitHub Actions 5-го числа о 08:37 UTC.
Повна методика: [docs/energy_map.md](../docs/energy_map.md).

## Запуск

Локально: клонувати репозиторій, виконати `python -m pip install -r requirements.txt`
і відкрити цей ноутбук. Потрібні pandas та numpy (залежність pandas).
У Colab один раз виконайте в окремій комірці:

```python
!git clone https://github.com/Tetiana-Mokliak/ukraine-electricity-market.git
%cd ukraine-electricity-market
%pip install -r requirements.txt
```

Це завантажує файли з GitHub, а не звертається до Energy Map API.
Для актуального знімка використовується шлях із `current.json`, а не назва з датою.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'scripts/prepare_energy_map.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Спочатку відкрийте каталог репозиторію за інструкцією вище.')
sys.path.insert(0, str(ROOT))
from scripts.prepare_energy_map import clean_outages, prepare

DATA_DIR = ROOT / 'data/energy_map'
state = json.loads((DATA_DIR / 'current.json').read_text(encoding='utf-8'))
source_path = DATA_DIR / state['file']
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == state['sha256']
df = pd.read_csv(source_path, dtype='string')

## Джерело та свіжість

Дата останнього запису й дата перевірки джерела — різні показники.
Дата запуску цього ноутбука не є датою API-перевірки.

In [2]:
pd.Series({
    'Остання дата даних': state['data_end_date'],
    'Остання успішна перевірка API (UTC)': state['last_checked_at_utc'],
    'Рядків у джерелі': len(df),
    'Порожніх дат у джерелі': int(df['date'].isna().sum()),
    'SHA-256': state['sha256'],
})

Остання дата даних                                                            2026-06-30
Остання успішна перевірка API (UTC)                     2026-10-21T15:38:00.392808+00:00
Рядків у джерелі                                                                   30647
Порожніх дат у джерелі                                                               328
SHA-256                                8f5f35d119a9453172e7ee5a48e1a7a08241d0077fa3d8...
dtype: object

## Очищення

Спільна функція `clean_outages()` містить правила, написані під час навчання:
- «ні» + невідомий обсяг → 0; «так» + невідомий обсяг залишається невідомим;
- `number` → `outage_coverage` (географія); при «ні» порожнє значення → «не застосовувались»;
- точне число → однакові min/max, «від A до B» → A/B, «до B» → 0/B;
- 0/0 — «Без обмежень»; (0; 1] — «Низька»; (1; 2] — «Помірна»;
  min > 2 — «Висока»; перетин рівнів — «Змінна»; невідомі межі — «Невідомо».

Це **авторська шкала**, не офіційна оцінка й не тривалість відсутності світла за адресою.
Нуль для нижньої межі «до» — аналітична домовленість.

### Припущення про квітневий дефект

328 порожніх дат між 31.03 та 14.04.2026 відновлено за порядком рядків,
з перевіркою меж, кількості та послідовності Excel-посилань.
Постачальник відповідність кожного рядка годині **не підтверджував**.
Такі рядки мають `datetime_reconstructed=True`; їх можна виключити для аналізу чутливості.
Якщо джерело виправлене, реконструкція не застосовується. Нові дефекти зупиняють обробку.

Функції парсингу й категоризації та перевірки: `scripts/prepare_energy_map.py`.
Ноутбук і автоматичний запуск користуються одним кодом.

In [3]:
df_clean = clean_outages(df)
df_clean.head()

,date,hour,outage_use,outage_amount,outage_coverage,datetime_reconstructed,outage_amount_min,outage_amount_max,outage_intensity
0,2023-01-01,00:00-01:00,ні,0,не застосовувались,False,0.0,0.0,Без обмежень
1,2023-01-01,01:00-02:00,ні,0,не застосовувались,False,0.0,0.0,Без обмежень
2,2023-01-01,02:00-03:00,ні,0,не застосовувались,False,0.0,0.0,Без обмежень
3,2023-01-01,03:00-04:00,ні,0,не застосовувались,False,0.0,0.0,Без обмежень
4,2023-01-01,04:00-05:00,ні,0,не застосовувались,False,0.0,0.0,Без обмежень


## Перевірки

`clean_outages()` перевіряє календар і точний склад годин за Europe/Kyiv.
Повторну осінню годину зберігаємо; весняна доба має 23 години.

In [4]:
calendar = pd.date_range(df_clean['date'].min(), df_clean['date'].max())
assert calendar.difference(df_clean['date']).empty
assert len(df_clean) == len(df)
assert df_clean['date'].notna().all()
rows_per_day = df_clean.groupby('date').size()
rows_per_day[rows_per_day != 24]

date
2023-03-26    23
2023-10-29    25
2024-03-31    23
2024-10-27    25
2025-03-30    23
2025-10-26    25
2026-03-29    23
dtype: int64

In [5]:
df_clean.loc[df_clean['datetime_reconstructed'],
             ['outage_use', 'outage_intensity']].value_counts()

outage_use  outage_intensity
ні          Без обмежень        278
так         Невідомо             50
Name: count, dtype: int64

In [6]:
df_clean['outage_intensity'].value_counts(dropna=False)

outage_intensity
Без обмежень    24527
Невідомо         3831
Змінна            894
Висока            482
Помірна           457
Низька            456
Name: count, dtype: int64

## Збереження для дашборду

`prepare()` повторно використовує ту саму обробку й зберігає очищений CSV та аудит.
Незмінний уже оброблений знімок пропускається. Повтор після зміни методики
контролює `PROCESSOR_VERSION` у модулі. Запитів до API тут немає.
При помилці перевірки попередні результати залишаються доступними.

In [7]:
prepare(DATA_DIR)
print('CSV: data/energy_map/processed/outages.csv')
print('Аудит: data/energy_map/processed/audit.json')

Energy Map preparation: unchanged; skipped (no API requests).
CSV: data/energy_map/processed/outages.csv
Аудит: data/energy_map/processed/audit.json


## Обмеження використання

Це історія масштабного застосування графіків, а не фактичні години без світла за адресами.
Локальні ГПВ у 1–3 областях не враховуються. За 10.12.2025–28.01.2026
дані узагальнені на основі інформації обленерго. Постійний тримісячний лаг не підтверджений.
На дашборді та при поширенні даних потрібне посилання на
[Energy Map](https://energy-map.info/uk/datasets/0f8f9882-1fb2-47c6-81dc-31fbad914f16).
Токен зберігається лише в GitHub Actions secret `ENERGY_MAP_TOKEN`.
ENTSO-E цим ноутбуком не підключається.